# BharatGen → GGUF Converter

Converts `bharatgenai/LegalParam` or `bharatgenai/FinanceParam` (custom `ParamBharatGenForCausalLM` architecture) to a standard `LlamaForCausalLM` model, then uploads it to your HuggingFace account so the **gguf-my-repo** Space can quantize it to Q4_K_M.

**Runtime:** Runtime → Change runtime type → **T4 GPU** (free tier)

**Total time:** ~25–35 minutes

---

### Why this is needed
BharatGen wraps a standard LLaMA 2 architecture under a renamed class (`ParamBharatGenForCausalLM`). The `hf-to-gguf` converter rejects it by name. This notebook:
1. Loads the model with `trust_remote_code=True`
2. Re-saves it under the standard `LlamaForCausalLM` class (same weights, same tensors)
3. Uploads the clean version to your HF account
4. You then run `gguf-my-repo` on the clean version → perfect Q4_K_M output

## ⚙️ Step 0 — Configuration
**Edit only this cell.** Everything else runs automatically.

In [ ]:
# ─── EDIT THESE ───────────────────────────────────────────────────────────────

# Which BharatGen model to convert?
# Options: "bharatgenai/LegalParam"  or  "bharatgenai/FinanceParam"
SOURCE_MODEL_ID = "bharatgenai/LegalParam"

# Your HuggingFace username (the output repo will be created here)
HF_USERNAME = "your-hf-username"   # e.g. "atulgrover"

# Name for the output repo on HuggingFace
# This is what you paste into gguf-my-repo after this notebook finishes
OUTPUT_REPO_NAME = "LegalParam-Llama"   # or "FinanceParam-Llama"

# Private repo? Recommended: True
PRIVATE_REPO = True

# ─── DO NOT EDIT BELOW ────────────────────────────────────────────────────────
OUTPUT_REPO_ID = f"{HF_USERNAME}/{OUTPUT_REPO_NAME}"
LOCAL_SAVE_PATH = f"/content/{OUTPUT_REPO_NAME}"

print(f"Source : {SOURCE_MODEL_ID}")
print(f"Output : {OUTPUT_REPO_ID}")
print(f"Local  : {LOCAL_SAVE_PATH}")

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q \
    transformers>=4.40.0 \
    accelerate \
    torch \
    huggingface_hub \
    safetensors \
    sentencepiece \
    protobuf

print("✓ Dependencies installed")

## 🔑 Step 2 — HuggingFace Login
You need a **write-access token** to upload the converted model.

Get one at: **Settings → Access Tokens → New token → Write**

→ [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import login, HfApi

# This will prompt you to enter your HF token securely
login()

api = HfApi()
whoami = api.whoami()
print(f"✓ Logged in as: {whoami['name']}")

## 🔍 Step 3 — Inspect Source Config
Confirms the model architecture and verifies the tensor mapping will work.

In [ ]:
from transformers import AutoConfig
import json

print(f"Loading config from {SOURCE_MODEL_ID}...")
src_config = AutoConfig.from_pretrained(SOURCE_MODEL_ID, trust_remote_code=True)

print("\nSource model config:")
print(f"  Architecture:        {src_config.architectures}")
print(f"  Hidden size:         {src_config.hidden_size}")
print(f"  Num hidden layers:   {src_config.num_hidden_layers}")
print(f"  Num attention heads: {src_config.num_attention_heads}")
print(f"  Num KV heads:        {src_config.num_key_value_heads}")
print(f"  Intermediate size:   {src_config.intermediate_size}")
print(f"  Max position emb:    {src_config.max_position_embeddings}")
print(f"  RMS norm eps:        {src_config.rms_norm_eps}")
print(f"  RoPE theta:          {src_config.rope_theta}")
print(f"  Vocab size:          {src_config.vocab_size}")
print(f"  Hidden act:          {src_config.hidden_act}")
print(f"  BOS token id:        {src_config.bos_token_id}")
print(f"  EOS token id:        {src_config.eos_token_id}")

## ⬇️ Step 4 — Load Source Model
~5.8 GB download. Takes 5–10 minutes depending on HF speed.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading model: {SOURCE_MODEL_ID}")
print("(~5.8 GB download — takes 5–10 minutes)\n")

src_model = AutoModelForCausalLM.from_pretrained(
    SOURCE_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

src_tokenizer = AutoTokenizer.from_pretrained(
    SOURCE_MODEL_ID,
    trust_remote_code=True,
)

print(f"\n✓ Model loaded")
print(f"  Parameters: {sum(p.numel() for p in src_model.parameters()):,}")
print(f"  dtype:      {next(src_model.parameters()).dtype}")

## 🔄 Step 5 — Remap to Standard LlamaForCausalLM
Builds a standard Llama config from the source config values, creates a fresh LlamaForCausalLM, and copies all weights directly. The tensor names are identical — BharatGen only changed the class name.

In [ ]:
from transformers import LlamaConfig, LlamaForCausalLM

print("Building standard LlamaConfig from source config values...")

llama_config = LlamaConfig(
    hidden_size=src_config.hidden_size,
    intermediate_size=src_config.intermediate_size,
    num_hidden_layers=src_config.num_hidden_layers,
    num_attention_heads=src_config.num_attention_heads,
    num_key_value_heads=src_config.num_key_value_heads,
    hidden_act=src_config.hidden_act,
    max_position_embeddings=src_config.max_position_embeddings,
    rms_norm_eps=src_config.rms_norm_eps,
    rope_theta=src_config.rope_theta,
    rope_scaling=getattr(src_config, 'rope_scaling', None),
    vocab_size=src_config.vocab_size,
    bos_token_id=src_config.bos_token_id,
    eos_token_id=src_config.eos_token_id,
    pad_token_id=getattr(src_config, 'pad_token_id', None),
    tie_word_embeddings=getattr(src_config, 'tie_word_embeddings', False),
    attention_bias=getattr(src_config, 'attention_bias', False),
    mlp_bias=getattr(src_config, 'mlp_bias', False),
    torch_dtype="float16",
)

print("Initialising standard LlamaForCausalLM (empty weights)...")
llama_model = LlamaForCausalLM(llama_config)

print("Copying weights from source model...")
src_state = src_model.state_dict()
dst_state = llama_model.state_dict()

# Check tensor name alignment before copying
src_keys = set(src_state.keys())
dst_keys = set(dst_state.keys())
missing  = dst_keys - src_keys
extra    = src_keys - dst_keys

if missing:
    print(f"\n⚠️  Keys in target but not in source ({len(missing)}):")
    for k in sorted(missing)[:10]: print(f"    {k}")
if extra:
    print(f"\n⚠️  Keys in source but not in target ({len(extra)}):")
    for k in sorted(extra)[:10]: print(f"    {k}")

if not missing and not extra:
    print("✓ Tensor keys match perfectly — direct copy")
    llama_model.load_state_dict(src_state, strict=True)
else:
    print("\nAttempting partial load (strict=False)...")
    result = llama_model.load_state_dict(src_state, strict=False)
    print(f"  Missing keys: {len(result.missing_keys)}")
    print(f"  Unexpected:   {len(result.unexpected_keys)}")
    if result.missing_keys:
        print("  ⛔ STOP — critical tensors missing. Check key mapping.")
        for k in result.missing_keys: print(f"    {k}")
    else:
        print("  ✓ All target tensors populated — safe to continue")

print(f"\n✓ Weight copy complete")

# Free source model memory
del src_model
torch.cuda.empty_cache()
print("✓ Source model freed from memory")

## 🧪 Step 6 — Sanity Check (Quick Inference Test)
Runs a short forward pass to confirm the converted model generates coherent text **before** uploading.

In [ ]:
from transformers import pipeline

print("Running sanity check inference...")
llama_model = llama_model.cuda().half()

pipe = pipeline(
    "text-generation",
    model=llama_model,
    tokenizer=src_tokenizer,
    device=0,
    torch_dtype=torch.float16,
)

prompt = "[INST] What is the limitation period under Section 7 of the Insolvency and Bankruptcy Code, 2016? [/INST]"

output = pipe(
    prompt,
    max_new_tokens=120,
    do_sample=False,
    temperature=1.0,
    repetition_penalty=1.1,
)[0]["generated_text"]

print("\n" + "─" * 60)
print("PROMPT:", prompt)
print("─" * 60)
print("RESPONSE:", output[len(prompt):])
print("─" * 60)
print("\n✓ Sanity check passed if response is coherent legal text")
print("✗ STOP if response is gibberish or pure repetition")

# Move back to CPU before saving
llama_model = llama_model.cpu().float()

## 💾 Step 7 — Save Converted Model Locally

In [ ]:
import os

print(f"Saving to {LOCAL_SAVE_PATH} ...")
os.makedirs(LOCAL_SAVE_PATH, exist_ok=True)

llama_model.save_pretrained(
    LOCAL_SAVE_PATH,
    safe_serialization=True,   # saves as .safetensors (gguf-my-repo prefers this)
)
src_tokenizer.save_pretrained(LOCAL_SAVE_PATH)

# List saved files
saved_files = os.listdir(LOCAL_SAVE_PATH)
print(f"\n✓ Saved {len(saved_files)} files:")
for f in sorted(saved_files):
    size_mb = os.path.getsize(os.path.join(LOCAL_SAVE_PATH, f)) / (1024**2)
    print(f"  {f:50s}  {size_mb:7.1f} MB")

## ☁️ Step 8 — Upload to HuggingFace
Creates a private repo under your HF account and uploads the converted model.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

print(f"Creating repo: {OUTPUT_REPO_ID} (private={PRIVATE_REPO})")
api.create_repo(
    repo_id=OUTPUT_REPO_ID,
    private=PRIVATE_REPO,
    exist_ok=True,
)

print(f"Uploading to {OUTPUT_REPO_ID} ...")
print("(~6 GB upload — takes 10–15 minutes)\n")

api.upload_folder(
    folder_path=LOCAL_SAVE_PATH,
    repo_id=OUTPUT_REPO_ID,
    commit_message=f"Convert {SOURCE_MODEL_ID} → standard LlamaForCausalLM for GGUF conversion",
)

print(f"\n✓ Upload complete!")
print(f"   Repo URL: https://huggingface.co/{OUTPUT_REPO_ID}")

## ✅ Step 9 — Next: Run gguf-my-repo

Your converted model is now on HuggingFace. Now run the automated GGUF converter:

In [ ]:
print("=" * 65)
print(" NEXT STEP")
print("=" * 65)
print()
print(" 1. Open: https://huggingface.co/spaces/ggml-org/gguf-my-repo")
print()
print(f" 2. Hub Model ID:            {OUTPUT_REPO_ID}")
print( "    GGML Quantization Type:   Q4_K_M")
print( "    Private Repo:             ✓ checked")
print()
print( " 3. Click Submit. Wait ~10–15 minutes.")
print()
print(f" 4. Download the .gguf from: https://huggingface.co/{OUTPUT_REPO_ID}-GGUF")
print()
print( " 5. Copy to Hayagriva models path:")
print( "    ~/Library/Application Support/Hayagriva/models/legalparam.gguf")
print()
print( " 6. Test with llama-server:")
print( "    brew install llama.cpp")
print( "    llama-server --model ~/Library/Application\\ Support/Hayagriva/models/legalparam.gguf \\")
print( "      --port 8080 --ctx-size 2048 --n-gpu-layers 99")
print("=" * 65)

---

## 🧪 Bonus — Local GGUF Verification (run after downloading the .gguf)

After downloading the Q4_K_M `.gguf` file from HuggingFace, test it from your Mac terminal:

```bash
# Install llama-server (one-time, Metal GPU support included)
brew install llama.cpp

# Start inference server
llama-server \
  --model ~/Library/Application\ Support/Hayagriva/models/legalparam.gguf \
  --port 8080 \
  --ctx-size 2048 \
  --n-gpu-layers 99

# In a new terminal — test with a legal prompt
curl http://127.0.0.1:8080/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{
    "model": "legalparam",
    "messages": [
      {"role": "system", "content": "You are an expert in Indian insolvency law under IBC 2016."},
      {"role": "user", "content": "What is the limitation period for filing under Section 7 of the IBC? Cite any Supreme Court ruling."}
    ],
    "max_tokens": 300,
    "temperature": 0.1
  }'
```

**Good output ✅:** Mentions 3-year limitation, IBC Section 7, possible SC case citation

**Bad output ❌:** Repetition loops, wrong act citations, switching languages randomly

### Expected performance on M-series Mac:
| Chip | RAM | Tokens/sec |
|---|---|---|
| M1 16GB | Apple GPU (Metal) | ~12–16 t/s |
| M2 16GB | Apple GPU (Metal) | ~18–22 t/s |
| M3 16GB | Apple GPU (Metal) | ~22–28 t/s |
| Intel Mac | CPU only | ~2–4 t/s (very slow) |